# Wright-Fisher Diffusion: CTMC Approximation with Equal Escape Rates

This notebook develops a continuous-time Markov chain (CTMC) approximation to the Wright-Fisher diffusion process. The key insight is that **bin widths proportional to the local standard deviation equalize escape rates across all states**, and this property holds regardless of selection.

## 1. The Wright-Fisher Diffusion

The Wright-Fisher model describes allele frequency change in a finite population. In the diffusion limit ($N \to \infty$, $s \to 0$, $Ns$ constant), the allele frequency $p$ evolves according to the stochastic differential equation:

$$dp = \mu(p)\,dt + \sigma(p)\,dW$$

where $W$ is a standard Wiener process, and:

**Drift (directional force from selection):**
$$\mu(p) = sp(1-p)\left[h + (1-2h)p\right]$$

**Infinitesimal variance (genetic drift):**
$$\sigma^2(p) = \frac{p(1-p)}{2N_e}$$

Here:
- $N_e$ = effective population size
- $s$ = selection coefficient
- $h$ = dominance coefficient ($h=0.5$ for additive selection)

The boundaries $p=0$ (loss) and $p=1$ (fixation) are **absorbing states**.

## 2. CTMC Discretization

We approximate the continuous state space $[0,1]$ with discrete bins. Let bin $i$ span $[p_i, p_{i+1}]$ with width $\Delta_i = p_{i+1} - p_i$ and midpoint $\bar{p}_i$.

### Transition rates from the Fokker-Planck equation

The standard discretization of a diffusion process gives transition rates:

$$\lambda_{i \to i+1} = \frac{\sigma^2(\bar{p}_i)}{2\Delta_i^2} + \frac{\mu(\bar{p}_i)}{2\Delta_i}$$

$$\lambda_{i \to i-1} = \frac{\sigma^2(\bar{p}_i)}{2\Delta_i^2} - \frac{\mu(\bar{p}_i)}{2\Delta_i}$$

The first term represents diffusion (symmetric random motion), while the second represents drift (directional bias from selection).

## 3. Key Result: Escape Rate Independence from Selection

The **total escape rate** from state $i$ is:

$$\lambda_i^{\text{out}} = \lambda_{i \to i+1} + \lambda_{i \to i-1}$$

Substituting:

$$\lambda_i^{\text{out}} = \left[\frac{\sigma^2}{2\Delta_i^2} + \frac{\mu}{2\Delta_i}\right] + \left[\frac{\sigma^2}{2\Delta_i^2} - \frac{\mu}{2\Delta_i}\right] = \frac{\sigma^2(\bar{p}_i)}{\Delta_i^2}$$

**The drift terms cancel!** The escape rate depends only on the local variance, not on selection.

### Physical interpretation

Selection determines *which direction* you're likely to exit, but not *how quickly* you exit. The total rate of leaving a bin is purely a function of the local diffusion strength.

This is analogous to a biased random walk: the bias affects where you end up, but the rate at which you take steps is independent of the bias.

## 4. Equal Escape Rate Bins

To equalize escape rates across all bins, we require:

$$\frac{\sigma^2(\bar{p}_i)}{\Delta_i^2} = \text{constant}$$

This means:

$$\Delta_i \propto \sigma(\bar{p}_i) = \sqrt{\frac{\bar{p}_i(1-\bar{p}_i)}{2N_e}} \propto \sqrt{\bar{p}_i(1-\bar{p}_i)}$$

### The arcsine transformation

Consider the transformation:

$$\theta = \arcsin(\sqrt{p})$$

By the chain rule:

$$\frac{d\theta}{dp} = \frac{1}{2\sqrt{p(1-p)}}$$

So equal intervals in $\theta$-space correspond to intervals in $p$-space with:

$$\Delta p \approx \Delta\theta \cdot 2\sqrt{p(1-p)}$$

This is exactly what we need! 

### Bin boundaries

For $n$ bins, we divide $\theta \in [0, \pi/2]$ uniformly:

$$\theta_i = \frac{i\pi}{2n}, \quad i = 0, 1, \ldots, n$$

The bin boundaries in frequency space are:

$$\boxed{p_i = \sin^2\left(\frac{i\pi}{2n}\right)}$$

## 5. Why Selection Doesn't Change the Optimal Binning

We've shown that the escape rate $\lambda^{\text{out}} = \sigma^2/\Delta^2$ is independent of $\mu$. Let's verify this doesn't break down in edge cases.

### Constraint: Numerical stability (Péclet number)

The discretization is accurate when diffusion dominates drift within each bin. The **Péclet number** measures this:

$$\text{Pe}_i = \frac{|\mu(\bar{p}_i)| \cdot \Delta_i}{\sigma^2(\bar{p}_i)}$$

For the CTMC to be a good approximation, we need $\text{Pe} \lesssim 1$.

### Additive selection ($h = 1/2$)

$$\mu(p) = \frac{s}{2}p(1-p)$$

The Péclet number becomes:

$$\text{Pe} = \frac{|s/2| \cdot p(1-p) \cdot \Delta}{p(1-p)/(2N_e)} = |s| N_e \cdot \Delta$$

This is **independent of $p$** — so arcsine bins remain optimal. We just need enough bins that $\Delta_{\max} < 1/(|s|N_e)$.

### Non-additive selection ($h \neq 1/2$)

$$\mu(p) = sp(1-p)[h + (1-2h)p]$$

The Péclet number:

$$\text{Pe} = 2|s|N_e \cdot |h + (1-2h)p| \cdot \Delta$$

Now $\text{Pe}$ varies with $p$. For strongly non-additive selection, you might need adaptive refinement. But for most practical cases with $|h - 0.5| \lesssim 0.3$, the arcsine bins work well with sufficient $n$.

## 6. Implementation

In [ ]:
import numpy as np
from scipy.linalg import expm
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Tuple, Optional

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

In [ ]:
@dataclass
class WrightFisherCTMC:
    """Wright-Fisher diffusion approximated by a CTMC."""
    n_bins: int
    Ne: float
    s: float
    h: float
    boundaries: np.ndarray  # length n_bins + 1
    midpoints: np.ndarray   # length n_bins
    Q: np.ndarray           # rate matrix (n_bins + 2) x (n_bins + 2)
    
    @property
    def n_states(self) -> int:
        return self.n_bins + 2
    
    def state_to_freq(self, state: int) -> float:
        """Convert state index to frequency."""
        if state == 0:
            return 0.0
        elif state == self.n_bins + 1:
            return 1.0
        else:
            return self.midpoints[state - 1]
    
    def freq_to_state(self, p: float) -> int:
        """Find state containing frequency p."""
        if p <= 0:
            return 0
        elif p >= 1:
            return self.n_bins + 1
        else:
            return np.searchsorted(self.boundaries[1:], p) + 1

In [ ]:
def equal_escape_rate_bins(n_bins: int, p_min: float = 1e-6, p_max: float = 1 - 1e-6
                          ) -> Tuple[np.ndarray, np.ndarray]:
    """
    Create bins with equal escape rates using the arcsine transformation.
    
    Boundaries: p_i = sin²(iπ/2n)
    """
    theta_min = np.arcsin(np.sqrt(p_min))
    theta_max = np.arcsin(np.sqrt(p_max))
    
    theta = np.linspace(theta_min, theta_max, n_bins + 1)
    boundaries = np.sin(theta)**2
    
    # Midpoints computed in theta-space for consistency
    theta_mid = (theta[:-1] + theta[1:]) / 2
    midpoints = np.sin(theta_mid)**2
    
    return boundaries, midpoints

In [ ]:
def build_rate_matrix(boundaries: np.ndarray, midpoints: np.ndarray, 
                      Ne: float, s: float = 0.0, h: float = 0.5) -> np.ndarray:
    """
    Build the generator matrix Q for the CTMC.
    
    States: 0 = loss (absorbing)
            1 to n_bins = transient
            n_bins + 1 = fixation (absorbing)
    """
    n_bins = len(midpoints)
    n_states = n_bins + 2
    Q = np.zeros((n_states, n_states))
    
    for i in range(n_bins):
        state = i + 1
        p = midpoints[i]
        delta = boundaries[i + 1] - boundaries[i]
        
        # Local variance and drift
        variance = p * (1 - p) / (2 * Ne)
        drift = s * p * (1 - p) * (h + (1 - 2*h) * p)
        
        # Transition rates
        rate_up = variance / (2 * delta**2) + drift / (2 * delta)
        rate_down = variance / (2 * delta**2) - drift / (2 * delta)
        
        # Ensure non-negative (can happen with strong selection)
        rate_up = max(0, rate_up)
        rate_down = max(0, rate_down)
        
        # Fill matrix
        if i == 0:
            Q[state, 0] = rate_down          # to loss
            Q[state, state + 1] = rate_up   # to next bin
        elif i == n_bins - 1:
            Q[state, state - 1] = rate_down # to previous bin
            Q[state, n_states - 1] = rate_up # to fixation
        else:
            Q[state, state - 1] = rate_down
            Q[state, state + 1] = rate_up
        
        Q[state, state] = -(rate_up + rate_down)
    
    return Q

In [ ]:
def build_wf_ctmc(n_bins: int, Ne: float, s: float = 0.0, h: float = 0.5,
                  p_min: float = 1e-6, p_max: float = 1 - 1e-6) -> WrightFisherCTMC:
    """Construct a Wright-Fisher CTMC with equal escape rate bins."""
    boundaries, midpoints = equal_escape_rate_bins(n_bins, p_min, p_max)
    Q = build_rate_matrix(boundaries, midpoints, Ne, s, h)
    
    return WrightFisherCTMC(
        n_bins=n_bins, Ne=Ne, s=s, h=h,
        boundaries=boundaries, midpoints=midpoints, Q=Q
    )

## 7. Verification: Equal Escape Rates

In [ ]:
def get_escape_rates(mc: WrightFisherCTMC) -> np.ndarray:
    """Extract escape rates from diagonal of Q."""
    return -np.diag(mc.Q)[1:-1]

def get_peclet_numbers(mc: WrightFisherCTMC) -> np.ndarray:
    """Compute Péclet number for each bin."""
    widths = np.diff(mc.boundaries)
    peclet = np.zeros(mc.n_bins)
    
    for i, p in enumerate(mc.midpoints):
        variance = p * (1 - p) / (2 * mc.Ne)
        drift = abs(mc.s * p * (1 - p) * (mc.h + (1 - 2*mc.h) * p))
        peclet[i] = drift * widths[i] / variance if variance > 0 else 0
    
    return peclet

In [ ]:
# Build neutral model
mc_neutral = build_wf_ctmc(n_bins=20, Ne=1000, s=0.0)

# Check escape rates
escape_rates = get_escape_rates(mc_neutral)

print("Neutral case (s=0):")
print(f"  Escape rate mean: {np.mean(escape_rates):.6f}")
print(f"  Escape rate std:  {np.std(escape_rates):.2e}")
print(f"  CV:               {np.std(escape_rates)/np.mean(escape_rates):.2e}")

In [ ]:
# Build model with selection
mc_selection = build_wf_ctmc(n_bins=20, Ne=1000, s=0.01, h=0.5)

escape_rates_sel = get_escape_rates(mc_selection)

print("With selection (s=0.01, h=0.5):")
print(f"  Escape rate mean: {np.mean(escape_rates_sel):.6f}")
print(f"  Escape rate std:  {np.std(escape_rates_sel):.2e}")
print(f"  CV:               {np.std(escape_rates_sel)/np.mean(escape_rates_sel):.2e}")
print(f"\nEscape rates identical: {np.allclose(escape_rates, escape_rates_sel)}")

In [ ]:
# Visualize the rate matrix structure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Neutral
ax1 = axes[0]
im1 = ax1.imshow(mc_neutral.Q, cmap='RdBu_r', aspect='equal')
ax1.set_title('Rate matrix Q (neutral)')
ax1.set_xlabel('To state')
ax1.set_ylabel('From state')
plt.colorbar(im1, ax=ax1, shrink=0.8)

# With selection
ax2 = axes[1]
im2 = ax2.imshow(mc_selection.Q, cmap='RdBu_r', aspect='equal')
ax2.set_title('Rate matrix Q (s=0.01)')
ax2.set_xlabel('To state')
ax2.set_ylabel('From state')
plt.colorbar(im2, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.show()

## 8. Visualizing the Bin Structure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

mc = mc_neutral

# Panel 1: Bin boundaries on frequency axis
ax1 = axes[0, 0]
widths = np.diff(mc.boundaries)
colors = plt.cm.viridis(np.linspace(0.2, 0.8, mc.n_bins))

for i in range(mc.n_bins):
    ax1.barh(0, widths[i], left=mc.boundaries[i], height=0.5, 
             color=colors[i], edgecolor='black', linewidth=0.5)
    
ax1.set_xlim(-0.02, 1.02)
ax1.set_ylim(-0.5, 0.5)
ax1.set_xlabel('Allele frequency p')
ax1.set_yticks([])
ax1.set_title('Bin structure (wider at p ≈ 0.5)')
ax1.axvline(0, color='blue', linewidth=2, label='Loss')
ax1.axvline(1, color='red', linewidth=2, label='Fixation')
ax1.legend(loc='upper right')

# Panel 2: Bin widths vs frequency
ax2 = axes[0, 1]
ax2.bar(mc.midpoints, widths, width=widths*0.8, alpha=0.7, edgecolor='black')
# Theoretical curve
p_theory = np.linspace(0.01, 0.99, 100)
width_theory = np.sqrt(p_theory * (1 - p_theory))
width_theory = width_theory / width_theory.sum() * widths.sum()  # normalize
ax2.plot(p_theory, width_theory, 'r-', linewidth=2, label=r'$\propto \sqrt{p(1-p)}$')
ax2.set_xlabel('Frequency p')
ax2.set_ylabel('Bin width Δp')
ax2.set_title('Bin widths scale with standard deviation')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Panel 3: Escape rates (should be flat)
ax3 = axes[1, 0]
ax3.bar(range(1, mc.n_bins + 1), escape_rates, alpha=0.7, edgecolor='black')
ax3.axhline(np.mean(escape_rates), color='red', linestyle='--', 
            label=f'Mean = {np.mean(escape_rates):.4f}')
ax3.set_xlabel('State index')
ax3.set_ylabel('Escape rate λ')
ax3.set_title('Escape rates (constant across states)')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Panel 4: Variance and bin boundaries
ax4 = axes[1, 1]
p_range = np.linspace(0.001, 0.999, 500)
variance = p_range * (1 - p_range) / (2 * mc.Ne)
ax4.plot(p_range, variance, 'b-', linewidth=2, label=r'$\sigma^2(p) = p(1-p)/2N_e$')

for b in mc.boundaries[1:-1]:
    ax4.axvline(b, color='gray', linestyle='--', alpha=0.5)

ax4.set_xlabel('Frequency p')
ax4.set_ylabel('Infinitesimal variance')
ax4.set_title('Variance peaks at p=0.5 → narrowest bins there')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Effect of Selection on Transition Asymmetry

In [ ]:
def get_transition_asymmetry(mc: WrightFisherCTMC) -> Tuple[np.ndarray, np.ndarray]:
    """Extract up and down rates for each transient state."""
    rates_up = np.zeros(mc.n_bins)
    rates_down = np.zeros(mc.n_bins)
    
    for i in range(mc.n_bins):
        state = i + 1
        if i == 0:
            rates_down[i] = mc.Q[state, 0]
            rates_up[i] = mc.Q[state, state + 1]
        elif i == mc.n_bins - 1:
            rates_down[i] = mc.Q[state, state - 1]
            rates_up[i] = mc.Q[state, mc.n_states - 1]
        else:
            rates_down[i] = mc.Q[state, state - 1]
            rates_up[i] = mc.Q[state, state + 1]
    
    return rates_up, rates_down

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

selection_values = [0, 0.005, 0.02]

for ax, s in zip(axes, selection_values):
    mc = build_wf_ctmc(n_bins=20, Ne=1000, s=s, h=0.5)
    rates_up, rates_down = get_transition_asymmetry(mc)
    
    x = np.arange(mc.n_bins)
    width = 0.35
    
    ax.bar(x - width/2, rates_up, width, label='Rate up (→ fixation)', alpha=0.7)
    ax.bar(x + width/2, rates_down, width, label='Rate down (→ loss)', alpha=0.7)
    ax.axhline(np.mean(rates_up + rates_down)/2, color='red', linestyle='--', 
               alpha=0.5, label='Mean rate')
    
    ax.set_xlabel('State index')
    ax.set_ylabel('Transition rate')
    ax.set_title(f's = {s} (Ns = {s*1000:.0f})')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Selection affects asymmetry, not total escape rate', y=1.02)
plt.tight_layout()
plt.show()

## 10. Computing Fixation Probabilities

In [ ]:
def fixation_probability(mc: WrightFisherCTMC) -> np.ndarray:
    """
    Compute fixation probability from each transient state.
    
    Solves: Q_T @ u = -r_fix, where Q_T is transient-to-transient block.
    """
    n = mc.n_bins
    Q_T = mc.Q[1:n+1, 1:n+1]
    r_fix = mc.Q[1:n+1, -1]
    
    return np.linalg.solve(Q_T, -r_fix)


def kimura_fixation_prob(p: np.ndarray, Ne: float, s: float) -> np.ndarray:
    """
    Kimura's (1962) fixation probability formula.
    
    P(fix) = (1 - e^{-4Ns p}) / (1 - e^{-4Ns})
    """
    if abs(s) < 1e-10:
        return p
    
    S = 4 * Ne * s
    return (1 - np.exp(-S * p)) / (1 - np.exp(-S))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

Ne = 1000
n_bins = 30

for s, color in [(0, 'black'), (0.001, 'blue'), (0.005, 'green'), (-0.005, 'red')]:
    mc = build_wf_ctmc(n_bins=n_bins, Ne=Ne, s=s)
    fix_prob = fixation_probability(mc)
    
    # CTMC result
    ax.plot(mc.midpoints, fix_prob, 'o', color=color, markersize=4, alpha=0.7)
    
    # Kimura's formula
    p_fine = np.linspace(0.001, 0.999, 200)
    kimura = kimura_fixation_prob(p_fine, Ne, s)
    label = f's={s} (Ns={Ne*s:.0f})' if s != 0 else 'Neutral'
    ax.plot(p_fine, kimura, '-', color=color, linewidth=2, label=label)

ax.set_xlabel('Initial frequency p')
ax.set_ylabel('Fixation probability')
ax.set_title('CTMC (dots) vs Kimura formula (lines)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 11. Transition Probability Matrix P(t) = exp(Qt)

In [ ]:
def transition_matrix(mc: WrightFisherCTMC, t: float) -> np.ndarray:
    """Compute P(t) = exp(Qt). Time in units of 2Ne generations."""
    return expm(mc.Q * t)

In [ ]:
mc = build_wf_ctmc(n_bins=20, Ne=1000, s=0)

# Starting from p ≈ 0.1
start_state = mc.freq_to_state(0.1)
print(f"Starting state: {start_state} (p ≈ {mc.state_to_freq(start_state):.3f})")

times = [0.1, 0.5, 1.0, 2.0, 5.0]

fig, axes = plt.subplots(1, len(times), figsize=(15, 3))

for ax, t in zip(axes, times):
    P = transition_matrix(mc, t)
    probs = P[start_state, :]
    
    # Plot distribution over states
    freqs = [mc.state_to_freq(i) for i in range(mc.n_states)]
    ax.bar(freqs, probs, width=0.03, alpha=0.7, edgecolor='black')
    
    # Highlight absorbing states
    ax.bar([0], [probs[0]], width=0.03, color='blue', alpha=0.9, label=f'Loss: {probs[0]:.2f}')
    ax.bar([1], [probs[-1]], width=0.03, color='red', alpha=0.9, label=f'Fix: {probs[-1]:.2f}')
    
    ax.set_xlabel('Frequency')
    ax.set_ylabel('Probability')
    ax.set_title(f't = {t}')
    ax.legend(fontsize=7)
    ax.set_xlim(-0.05, 1.05)

plt.suptitle(f'Evolution of frequency distribution from p₀ ≈ {mc.state_to_freq(start_state):.2f}', y=1.05)
plt.tight_layout()
plt.show()

## 12. Phase-Type Distribution of Absorption Time

The time to absorption (loss or fixation) follows a **phase-type distribution** defined by the transient block of Q.

If we start in transient state $i$, the absorption time $T$ has:
- CDF: $F(t) = 1 - \mathbf{e}_i^T \exp(Q_T t) \mathbf{1}$
- Mean: $E[T] = -\mathbf{e}_i^T Q_T^{-1} \mathbf{1}$

In [ ]:
def mean_absorption_time(mc: WrightFisherCTMC) -> np.ndarray:
    """Mean time to absorption from each transient state."""
    Q_T = mc.Q[1:-1, 1:-1]
    ones = np.ones(mc.n_bins)
    return -np.linalg.solve(Q_T, ones)


def absorption_time_cdf(mc: WrightFisherCTMC, start_state: int, times: np.ndarray) -> np.ndarray:
    """CDF of absorption time starting from given state."""
    Q_T = mc.Q[1:-1, 1:-1]
    state_idx = start_state - 1  # Convert to 0-indexed in transient block
    
    cdf = np.zeros(len(times))
    for i, t in enumerate(times):
        P_T = expm(Q_T * t)
        cdf[i] = 1 - P_T[state_idx, :].sum()
    
    return cdf

In [ ]:
mc = build_wf_ctmc(n_bins=30, Ne=1000, s=0)

mean_times = mean_absorption_time(mc)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Mean absorption time vs initial frequency
ax1 = axes[0]
ax1.plot(mc.midpoints, mean_times, 'b-', linewidth=2)
ax1.set_xlabel('Initial frequency p')
ax1.set_ylabel('Mean time to absorption (2Nₑ generations)')
ax1.set_title('Mean absorption time')
ax1.grid(True, alpha=0.3)

# CDF for different starting frequencies
ax2 = axes[1]
t_range = np.linspace(0, 5, 200)

for p0 in [0.1, 0.3, 0.5]:
    state = mc.freq_to_state(p0)
    cdf = absorption_time_cdf(mc, state, t_range)
    ax2.plot(t_range, cdf, linewidth=2, label=f'p₀ = {p0}')

ax2.set_xlabel('Time (2Nₑ generations)')
ax2.set_ylabel('P(absorbed by time t)')
ax2.set_title('Absorption time CDF')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 13. Summary

### Key results

1. **Equal escape rate bins** use widths $\Delta_i \propto \sqrt{p_i(1-p_i)}$, achieved via the arcsine transformation.

2. **Selection doesn't affect escape rates** — the drift terms cancel in $\lambda^{\text{out}} = \lambda_{\uparrow} + \lambda_{\downarrow}$.

3. **Same binning works for all selection strengths**, subject to having enough bins to satisfy the Péclet constraint.

4. **The CTMC accurately reproduces** fixation probabilities and other diffusion properties.

### Bin boundary formula

$$p_i = \sin^2\left(\frac{i\pi}{2n}\right), \quad i = 0, 1, \ldots, n$$

In [ ]:
# Quick reference function
def print_bin_boundaries(n_bins: int):
    """Print equal-escape-rate bin boundaries."""
    boundaries = np.sin(np.linspace(0, np.pi/2, n_bins + 1))**2
    print(f"Bin boundaries for {n_bins} bins:")
    print(f"  {boundaries}")
    return boundaries

_ = print_bin_boundaries(10)